# Nemotron ColEmbed V2 — production tunnel for Phase 1

**Purpose**: Expose Nemotron's image and query embedding methods as HTTP endpoints (via ngrok) so the local pipeline (`retriever/nemotron.py`) can call them from off-Colab.

**Phase**: 1 (vision_only / hybrid modes)

---

## How to run
1. **Runtime → Disconnect and delete runtime** (clean VRAM)
2. **Runtime → Change runtime type → T4 GPU** (free) or L4 (Pro)
3. Left sidebar 🔑 → add `HF_TOKEN` secret. Optional: `NGROK_TOKEN` for stable URLs.
4. Run cells 1 → 6 in order. Cell 6 will print a `COLAB_EMBEDDING_URL=...` line.
5. Copy that URL into your local `.env` file as `COLAB_EMBEDDING_URL=...`
6. **Leave this notebook running** while you use the local indexer/retriever scripts.

## Endpoints exposed by cell 6
| Method | Path | Body | Returns |
|---|---|---|---|
| GET | `/health` | — | `{status, model_id, dtype, vram_gb}` |
| POST | `/embed_image` | multipart `file` (image) | `{shape, dtype, embedding_b64}` |
| POST | `/embed_query` | JSON `{"query": "..."}` | `{shape, dtype, embedding_b64}` |
| POST | `/embed_queries` | JSON `{"queries": ["...", ...]}` | `{embeddings: [{shape, embedding_b64}, ...]}` |

Embeddings are float16 numpy bytes, base64-encoded. Decode on the client with `np.frombuffer(base64.b64decode(b64), dtype=np.float16).reshape(shape)`.

## 1. Install

In [ ]:
!apt-get install -y -qq poppler-utils
!pip install -q transformers accelerate einops sentencepiece pypdfium2 pdf2image Pillow
!pip install -q pyngrok fastapi 'uvicorn[standard]' nest-asyncio python-multipart

## 2. HuggingFace auth

In [ ]:
from huggingface_hub import login
from google.colab import userdata
import os

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
login(token=os.environ['HF_TOKEN'])

## 3. Load Nemotron ColEmbed V2 (bf16 + float-only cast)

In [ ]:
import torch
from transformers import AutoModel

MODEL_ID = 'nvidia/llama-nemotron-colembed-vl-3b-v2'   # T4-friendly
# MODEL_ID = 'nvidia/nemotron-colembed-vl-4b-v2'       # L4 / Pro
# MODEL_ID = 'nvidia/nemotron-colembed-vl-8b-v2'       # L4 / A100

device = 'cuda' if torch.cuda.is_available() else 'cpu'
vram_gb = (torch.cuda.get_device_properties(0).total_memory / 1e9) if torch.cuda.is_available() else 0
print(f'Device: {torch.cuda.get_device_name(0) if device=="cuda" else "cpu"}  VRAM: {vram_gb:.1f} GB')

model = AutoModel.from_pretrained(
    MODEL_ID,
    device_map=device,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
).eval()

FLOAT_DTYPES = (torch.float32, torch.float16, torch.bfloat16, torch.float64)
for p in model.parameters():
    if p.dtype in FLOAT_DTYPES:
        p.data = p.data.to(torch.bfloat16)
for b in model.buffers():
    if b.dtype in FLOAT_DTYPES:
        b.data = b.data.to(torch.bfloat16)

print(f'Model loaded. Params: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B')

## 4. Helper — embedding to base64

Embeddings are kept as float16 on the wire to halve bandwidth. The local client decodes them back to a numpy float16 array, then converts to float32 for Qdrant if needed.

In [ ]:
import base64

def emb_to_payload(t):
    """torch tensor -> {shape, dtype, embedding_b64}."""
    arr = t.detach().to(torch.float16).cpu().numpy()
    return {
        'shape': list(arr.shape),
        'dtype': 'float16',
        'embedding_b64': base64.b64encode(arr.tobytes()).decode(),
    }

def first_emb(out):
    """Normalize forward_images / forward_queries return into a single tensor per item."""
    if isinstance(out, list):
        return out
    if hasattr(out, 'dim') and out.dim() == 3:
        return [out[i] for i in range(out.shape[0])]
    return [out]

## 5. FastAPI app (health + image + query endpoints)

In [ ]:
import io
from fastapi import FastAPI, UploadFile, File, HTTPException
from PIL import Image

app = FastAPI(title='nemotron-colembed-tunnel')

@app.get('/health')
def health():
    return {
        'status': 'ok',
        'model_id': MODEL_ID,
        'dtype': str(next(model.parameters()).dtype),
        'device': str(next(model.parameters()).device),
        'vram_gb': round(vram_gb, 1),
    }

@app.post('/embed_image')
async def embed_image(file: UploadFile = File(...)):
    try:
        img = Image.open(io.BytesIO(await file.read())).convert('RGB')
        with torch.no_grad():
            out = model.forward_images([img], batch_size=1)
        embs = first_emb(out)
        return emb_to_payload(embs[0])
    except Exception as e:
        raise HTTPException(status_code=500, detail=f'{type(e).__name__}: {e}')

@app.post('/embed_query')
async def embed_query(payload: dict):
    q = payload.get('query')
    if not isinstance(q, str) or not q.strip():
        raise HTTPException(status_code=400, detail='Body must be {"query": "<text>"}')
    try:
        with torch.no_grad():
            out = model.forward_queries([q], batch_size=1)
        embs = first_emb(out)
        return emb_to_payload(embs[0])
    except Exception as e:
        raise HTTPException(status_code=500, detail=f'{type(e).__name__}: {e}')

@app.post('/embed_queries')
async def embed_queries(payload: dict):
    qs = payload.get('queries')
    if not isinstance(qs, list) or not all(isinstance(q, str) and q for q in qs):
        raise HTTPException(status_code=400, detail='Body must be {"queries": ["<text>", ...]}')
    try:
        with torch.no_grad():
            out = model.forward_queries(qs, batch_size=min(8, len(qs)))
        embs = first_emb(out)
        return {'embeddings': [emb_to_payload(e) for e in embs]}
    except Exception as e:
        raise HTTPException(status_code=500, detail=f'{type(e).__name__}: {e}')

print('FastAPI app defined. 4 endpoints: GET /health, POST /embed_image, POST /embed_query, POST /embed_queries')

## 6. Start ngrok tunnel + run server

**This cell blocks**. Once it prints `COLAB_EMBEDDING_URL=...`, copy that URL into your local `.env`. Leave the cell running while you use local scripts.

Optional: add `NGROK_TOKEN` to Colab secrets for stable URLs and longer sessions.

In [ ]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok

try:
    ngrok_token = userdata.get('NGROK_TOKEN')
    if ngrok_token:
        ngrok.set_auth_token(ngrok_token)
        print('ngrok auth token set (stable URLs).')
except Exception:
    print('No NGROK_TOKEN secret — using anonymous tunnel (URL changes each run, 2-hour limit).')

ngrok.kill()  # close any existing tunnels from prior runs
public_url = ngrok.connect(8000)
print(f'\nCOLAB_EMBEDDING_URL={public_url}\n')
print('Test from your local machine:')
print(f'  curl {public_url}/health\n')
print('Leave this cell running. Stop with the cell stop button when done.')

nest_asyncio.apply()
uvicorn.run(app, host='0.0.0.0', port=8000)